# NOTEBOOK 09.1: XÁC MINH & KIỂM ĐỊNH PHÂN QUYỀN TRUY CẬP FIRESTORE THỜI KHÓA BIỂU & PHÒNG HỌC (CHECKPOINT 4.4.2)
## HỆ THỐNG SMART EDUCATION CENTER (SMARTEDU - THCS)

Notebook này phân tích, mô phỏng và kiểm định giải pháp khắc phục triệt để cảnh báo `Missing or insufficient permissions` trên các collection `schedules` và `rooms` của Firestore.

### Mục tiêu giải pháp:
1. **Bảo mật Zero-Trust:** Cấu hình Security Rules nghiêm ngặt trên Firestore: chỉ cho phép các vai trò học thuật hợp lệ (`ADMIN`, `ACADEMIC_STAFF`, `TEACHER`, `STUDENT`, `PARENT`) và từ chối tuyệt đối `ACCOUNTANT`.
2. **Đồng bộ hóa truy vấn theo phạm vi (Role-Scoped Querying):** Đảm bảo Client truy vấn đúng phạm vi quyền hạn tương ứng với Security Rules.
3. **Khởi tạo an toàn (Auth-Profile-Role Ready Lifecycle):** Không khởi chạy listener trước khi Auth và User Profile sẵn sàng.
4. **Loại bỏ Local Fallback giả tạo:** Báo cáo lỗi chính xác khi không có quyền hoặc lỗi kết nối.

In [ ]:
import json

# 1. Mô phỏng Ma Trận Phân Quyền Firestore (Security Rules Simulation)
ROLES = ['ADMIN', 'OWNER', 'ACADEMIC_STAFF', 'TEACHER', 'STUDENT', 'PARENT', 'ACCOUNTANT']

def evaluate_firestore_rule(role, collection_name, operation='read'):
    is_signed_in = role in ROLES
    if not is_signed_in:
        return False, "Unauthenticated"
    
    is_admin = role in ['ADMIN', 'OWNER']
    is_academic_staff = role == 'ACADEMIC_STAFF'
    is_teacher = role == 'TEACHER'
    is_student = role == 'STUDENT'
    is_parent = role == 'PARENT'
    is_accountant = role == 'ACCOUNTANT'
    
    if collection_name in ['rooms', 'schedules']:
        if operation == 'read':
            allowed = is_signed_in and (is_admin or is_academic_staff or is_teacher or is_student or is_parent) and not is_accountant
            return allowed, "ALLOWED" if allowed else "DENIED: Missing or insufficient permissions"
        elif operation in ['create', 'update', 'delete', 'write']:
            allowed = is_signed_in and (is_admin or is_academic_staff)
            return allowed, "ALLOWED" if allowed else "DENIED: Insufficient write privileges"
            
    return False, "Unknown collection"

print("=== MA TRẬN ĐÁNH GIÁ QUYỀN ĐỌC FIRESTORE ===")
for r in ROLES:
    sch_allowed, sch_msg = evaluate_firestore_rule(r, 'schedules', 'read')
    rm_allowed, rm_msg = evaluate_firestore_rule(r, 'rooms', 'read')
    print(f"Role: {r:15} | schedules: {sch_msg:20} | rooms: {rm_msg:20}")

In [ ]:
# 2. Mô phỏng Truy vấn Dữ liệu Theo Phạm Vi (Client-Side Query Scoping)
sample_schedules = [
    {
        "id": "sch_6A1_toan_t2",
        "classId": "class_6A1",
        "teacherId": "TCH-2026-001",
        "subjectId": "toan",
        "roomId": "room_101",
        "dayOfWeek": "MON",
        "startTime": "08:00",
        "endTime": "09:30",
        "status": "ACTIVE"
    },
    {
        "id": "sch_6A2_van_t2",
        "classId": "class_6A2",
        "teacherId": "TCH-2026-004",
        "subjectId": "van",
        "roomId": "room_102",
        "dayOfWeek": "MON",
        "startTime": "08:00",
        "endTime": "09:30",
        "status": "ACTIVE"
    },
    {
        "id": "sch_7A1_anh_t3",
        "classId": "class_7A1",
        "teacherId": "TCH-2026-007",
        "subjectId": "anh",
        "roomId": "room_lab1",
        "dayOfWeek": "TUE",
        "startTime": "09:45",
        "endTime": "11:15",
        "status": "ACTIVE"
    }
]

def resolve_role_schedules(user_profile, all_schedules):
    role = user_profile.get('role')
    
    if role in ['ADMIN', 'OWNER', 'ACADEMIC_STAFF']:
        # Collection query
        return all_schedules
    elif role == 'TEACHER':
        teacher_id = user_profile.get('teacherId') or user_profile.get('employeeCode')
        # where('teacherId', '==', teacher_id)
        return [s for s in all_schedules if s.get('teacherId') == teacher_id]
    elif role == 'STUDENT':
        class_id = user_profile.get('classId')
        # where('classId', '==', class_id)
        return [s for s in all_schedules if s.get('classId') == class_id]
    elif role == 'PARENT':
        child_class_ids = user_profile.get('childClassIds', [])
        # where('classId', 'in', child_class_ids)
        return [s for s in all_schedules if s.get('classId') in child_class_ids]
    elif role == 'ACCOUNTANT':
        # Strictly no listener / empty
        return []
    return []

# Kiểm thử cho từng hồ sơ
profiles = [
    {"id": "u_adm", "role": "ADMIN", "name": "Admin Tổng"},
    {"id": "u_tch1", "role": "TEACHER", "name": "Thầy Trần Quốc Việt", "teacherId": "TCH-2026-001"},
    {"id": "u_stu1", "role": "STUDENT", "name": "Học sinh Lớp 6A1", "classId": "class_6A1"},
    {"id": "u_par1", "role": "PARENT", "name": "Phụ huynh", "childClassIds": ["class_6A1", "class_7A1"]},
    {"id": "u_acc", "role": "ACCOUNTANT", "name": "Kế toán viên"}
]

print("=== KIỂM THỬ TRUY VẤN SCOPED SCHEDULES ===")
for p in profiles:
    res = resolve_role_schedules(p, sample_schedules)
    print(f"User: {p['name']} ({p['role']}) -> Số tiết học nhận được: {len(res)}")
    for item in res:
        print(f"   - {item['id']} | Lớp: {item['classId']} | GV: {item['teacherId']} | Phòng: {item['roomId']}")

In [ ]:
# 3. Xác thực tính độc lập & Ngăn chặn Cảnh báo Runtime
# Khi ACCOUNTANT đăng nhập, không thực thi onSnapshot() -> 0 cảnh báo permission denied
acc_profile = {"id": "u_acc", "role": "ACCOUNTANT"}
acc_schedules = resolve_role_schedules(acc_profile, sample_schedules)
assert len(acc_schedules) == 0, "Accountant không được phép nhận thời khóa biểu"

# Khi TEACHER đăng nhập, chỉ nhận lịch dạy của chính mình
tch_profile = {"id": "u_tch", "role": "TEACHER", "teacherId": "TCH-2026-001"}
tch_schedules = resolve_role_schedules(tch_profile, sample_schedules)
assert len(tch_schedules) == 1 and tch_schedules[0]['id'] == 'sch_6A1_toan_t2', "Teacher nhận sai lịch học"

print("✓ Toàn bộ các kiểm định Scoped Queries và Rules RBAC đều đạt PASS 100%!")